# Receptor–ligand binding ($k_{\mathrm{on}}$, $k_{\mathrm{off}}$)

*others · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/others/receptor_binding)

How strongly and how long does an immune receptor (a T-cell receptor or an antibody) hold on to its target? Mass-action kinetics with an on-rate $k_{on}$ and an off-rate $k_{off}$ is solved for several receptor variants and a whole titration of ligand concentrations at once, in one $(B, C)$ tensor, and shown as an SPR-style sensorgram next to the binding isotherms.

**Vector form**

$$R + L \;\underset{k_{\mathrm{off}}}{\overset{k_{\mathrm{on}}}{\rightleftharpoons}}\; C,\qquad \frac{dC}{dt} = k_{\mathrm{on}}\,R\,L - k_{\mathrm{off}}\,C,\qquad R = R_T - C,\qquad K_D = \frac{k_{\mathrm{off}}}{k_{\mathrm{on}}}$$

**Index (tensor) form**

$$J_{bc} = k^{\mathrm{on}}_b\,R_{bc}\,L_{bc} - k^{\mathrm{off}}_b\,C_{bc} \qquad \dot C_{bc} = -\dot R_{bc} = J_{bc} \qquad \theta_{bc} = \frac{L_c}{L_c + K_{D,b}}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $k^{\mathrm{on}}_b,\ k^{\mathrm{off}}_b$ | `k_on, k_off` | (B,) | rate constants of receptor variant $b$ (weak / medium / high-affinity TCR, antibody) |
| $L_c$ | `L` | (C,) | ligand concentration $c$ of the titration (nM); the last one is the sensorgram |
| $R_{bc},\ C_{bc}$ | `y[0], y[1]` | (B, C) | free receptors and complexes for every variant and concentration |
| $L_{bc}$ | `free_ligand(L, C, depletion)` | (B, C) | free ligand: $L_c$, or $L_c - C_{bc}$ with depletion |
| $k^{\mathrm{on}}_b R_{bc} L_{bc}$ | `einsum('b,bc,bc->bc', k_on, R, L)` | (B, C) | no index summed: $k_b$ is broadcast over every concentration $c$ |
| $\dot C_{bc}$ | `rhs(y, L, k_on, k_off, depletion)` | (2, B, C) | right-hand side for RK4; $R$ and $C$ change by $\mp J$, so $R + C = R_T$ |
| $K_{D,b}$ | `k_off / k_on` | (B,) | dissociation constant: the ligand level that binds half the receptors |
| $\theta_{bc}$ | `L[None, :] / (L[None, :] + K_D[:, None])` | (B, C) | equilibrium occupancy (dashed isotherms) |

Binding needs a free receptor and a free ligand to meet, so its rate is $k_{on} R L$; unbinding happens to each complex on its own, at rate $k_{off} C$. At equilibrium the two balance and the bound fraction is $\theta = L/(L + K_D)$ with $K_D = k_{off}/k_{on}$: half the receptors are occupied when $L = K_D$ (the tick on each isotherm). Starting from empty receptors the approach is exponential, $C(t) = C_{eq}(1 - e^{-k_{obs} t})$ with $k_{obs} = k_{on} L + k_{off}$, which is how SPR (surface plasmon resonance) instruments measure $k_{on}$; after the buffer wash the complexes decay as $e^{-k_{off} t}$ with half-life $t_{1/2} = \ln 2 / k_{off}$, the dwell time. The variants span realistic ranges: T-cell receptors bind peptide–MHC weakly ($K_D \sim 1$–$10\,\mu$M, dwell times of seconds), an antibody binds tightly ($K_D \sim 0.1$ nM, dwell time of hours). Look at the dots on the isotherms: at low concentration $k_{obs} \approx k_{off}$ is small, so the antibody has not yet reached equilibrium when the wash starts. Why does $k_{off}$ matter so much for T cells? In the kinetic proofreading picture, a bound TCR must pass through several signalling steps (phosphorylations) before it triggers the cell, and everything is reset if the ligand falls off first. A complex that lives longer than those steps is far more likely to signal, so the T cell reads the dwell time $1/k_{off}$ rather than the occupancy alone, and can tell a foreign peptide from a self peptide that differs only modestly in affinity. Simplifications: one binding site per receptor (no avidity or rebinding), a well-mixed solution and no mass-transport limit; with depletion on, every complex removes one ligand from a closed volume and the equilibrium follows the quadratic solution instead of $L/(L + K_D)$.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/others/receptor_binding/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    names = tensors.variant_names()
    k_on, k_off = tensors.rate_constants(dev)  # (B,), (B,)
    L = tensors.ligand_concentrations(params["log10_L"], dev)  # (C,) nM, last entry = sensorgram
    depletion = params["depletion"] == "on"
    R_T = params["R_T"]
    y0 = tensors.initial_state(R_T, len(names), len(L), dev)  # (2, B, C)

    times, traj = solve.solve(y0, L, k_on, k_off, params["t_assoc"], params["t_dissoc"], depletion)
    theta = traj[:, 1] / R_T  # (T, B, C) fraction of receptors bound

    K_D = equation.dissociation_constant(k_on, k_off)  # (B,) nM
    L_fine = torch.logspace(-2, 4, 200, device=dev)
    if depletion:
        theta_eq = equation.occupancy_with_depletion(L_fine, K_D, R_T)  # (B, 200)
    else:
        theta_eq = equation.occupancy(L_fine, K_D)

    t_half = equation.half_life(k_off)
    i_wash = int(round(params["t_assoc"] / (times[1] - times[0]).item()))
    print(f"sensorgram ligand L = {L[-1].item():.3g} nM   ·   depletion {params['depletion']}")
    for b, name in enumerate(names):
        print(
            f"{name:<19} K_D = {K_D[b].item():9.3g} nM   t½ = {t_half[b].item():8.3g} s   "
            f"θ at wash = {theta[i_wash, b, -1].item():.3f}   θ at end = {theta[-1, b, -1].item():.3f}"
        )
    return plot.animate(times, theta, L, L_fine, theta_eq, names, params["t_assoc"], out_dir / "receptor_binding.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/others/receptor_binding/equation.py
r"""Physics → tensor translation.

A ligand L (an antigen, or a peptide–MHC) binds a receptor R on an immune cell and forms a complex C:

    R + L  ⇌  C        forward rate k_on R L,   backward rate k_off C

For a batch of receptor variants b and a set of ligand concentrations c the net binding flux is

    J_{bc} = k^{on}_b R_{bc} L_{bc} - k^{off}_b C_{bc}

and the two species change by the same amount in opposite directions:

    \dot C_{bc} = J_{bc},    \dot R_{bc} = -J_{bc}      ⇒   R_{bc} + C_{bc} = R_T  (receptors are conserved)

No index is summed here: every (b, c) pair is an independent experiment. The einsum calls below only
make the broadcasting explicit — a rate constant k_b is shared by every concentration c.
"""

import math

import torch


def free_ligand(L_total: torch.Tensor, C: torch.Tensor, depletion: bool) -> torch.Tensor:
    """L_{bc} = L_c - δ C_{bc}.   L_total: (C,), C: (B, C) → (B, C).

    δ = 0: ligand is in large excess (or keeps flowing in, as in SPR), so L_{bc} = L_c.
    δ = 1: ligand depletion — every complex formed removes one ligand from solution.
    """
    if depletion:
        return L_total - C  # (C,) broadcast over b, minus (B, C)
    return L_total.expand_as(C)


def flux(R: torch.Tensor, C: torch.Tensor, L: torch.Tensor, k_on: torch.Tensor, k_off: torch.Tensor) -> torch.Tensor:
    r"""J_{bc} = k^{on}_b R_{bc} L_{bc} - k^{off}_b C_{bc}.   R, C, L: (B, C);  k_on, k_off: (B,)."""
    binding = torch.einsum("b,bc,bc->bc", k_on, R, L)  # association: needs a free receptor AND a free ligand
    unbinding = torch.einsum("b,bc->bc", k_off, C)  # dissociation: each complex falls apart on its own
    return binding - unbinding


def rhs(y: torch.Tensor, L_total: torch.Tensor, k_on: torch.Tensor, k_off: torch.Tensor, depletion: bool):
    """dy/dt for the state y of shape (2, B, C): y[0] = free receptors R, y[1] = complexes C."""
    R, C = y[0], y[1]
    L = free_ligand(L_total, C, depletion)
    J = flux(R, C, L, k_on, k_off)
    return torch.stack([-J, J])


def dissociation_constant(k_on: torch.Tensor, k_off: torch.Tensor) -> torch.Tensor:
    """K_D = k_off / k_on (a concentration): the ligand level that fills half the receptors.   (B,)"""
    return k_off / k_on


def half_life(k_off: torch.Tensor) -> torch.Tensor:
    """t_½ = ln 2 / k_off: how long a complex lasts on average before half of them have fallen apart.   (B,)"""
    return math.log(2.0) / k_off


def observed_rate(L: torch.Tensor, k_on: torch.Tensor, k_off: torch.Tensor) -> torch.Tensor:
    """k_obs,bc = k^on_b L_c + k^off_b: the rate at which binding approaches equilibrium.   (B, C)"""
    return torch.einsum("b,c->bc", k_on, L) + k_off[:, None]


def occupancy(L: torch.Tensor, K_D: torch.Tensor) -> torch.Tensor:
    """Equilibrium fraction of receptors bound, θ_bc = L_c / (L_c + K_D,b)  (ligand in excess).   (B, C)"""
    return L[None, :] / (L[None, :] + K_D[:, None])


def occupancy_with_depletion(L_total: torch.Tensor, K_D: torch.Tensor, R_T: float) -> torch.Tensor:
    """Equilibrium θ_bc when ligand is depleted: k_on (R_T - C)(L_T - C) = k_off C, a quadratic in C.

    C² - s C + R_T L_T = 0  with  s = R_T + L_T + K_D   →   the smaller root (the physical one, C ≤ R_T, L_T):

    C = (s - √(s² - 4 R_T L_T)) / 2 = 2 R_T L_T / (s + √(s² - 4 R_T L_T))

    The second form is the same number but avoids subtracting two nearly equal floats.
    """
    s = R_T + L_total[None, :] + K_D[:, None]  # (B, C)
    C = 2.0 * R_T * L_total[None, :] / (s + torch.sqrt(s**2 - 4.0 * R_T * L_total[None, :]))
    return C / R_T


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/others/receptor_binding/tensors.py
"""Build the concrete tensors: rate constants of each receptor variant, ligand concentrations, initial state.

Units: concentrations in nM, times in s. Rate constants are quoted in the usual M⁻¹ s⁻¹ and converted here.
"""

import torch

NANO = 1e-9  # 1 nM = 1e-9 M

# name, k_on (M⁻¹ s⁻¹), k_off (s⁻¹) — typical orders of magnitude
VARIANTS = [
    ("TCR, weak", 1e4, 1e-1),
    ("TCR, medium", 5e4, 5e-2),
    ("TCR, high-affinity", 1e5, 5e-3),
    ("antibody", 1e6, 1e-4),
]


def variant_names() -> list[str]:
    return [name for name, _, _ in VARIANTS]


def rate_constants(device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """k_on (B,) in nM⁻¹ s⁻¹ and k_off (B,) in s⁻¹."""
    k_on = torch.tensor([kon for _, kon, _ in VARIANTS], device=device) * NANO  # M⁻¹ s⁻¹ → nM⁻¹ s⁻¹
    k_off = torch.tensor([koff for _, _, koff in VARIANTS], device=device)
    return k_on, k_off


def ligand_concentrations(log10_flow_uM: float, device: torch.device, per_decade: int = 4) -> torch.Tensor:
    """L_c in nM, shape (C,): a log-spaced titration from 0.01 nM to 10 µM, plus the sensorgram
    concentration as the LAST entry (that column is the one drawn as an SPR trace)."""
    low, high = -2.0, 4.0  # log10 of nM
    grid = torch.logspace(low, high, int((high - low) * per_decade) + 1, device=device)
    flow = torch.tensor([1e3 * 10**log10_flow_uM], device=device)  # µM → nM
    return torch.cat([grid, flow])


def initial_state(R_T: float, n_variants: int, n_conc: int, device: torch.device) -> torch.Tensor:
    """y0 of shape (2, B, C): all receptors free (R = R_T) and no complexes (C = 0) before the ligand arrives."""
    R = torch.full((n_variants, n_conc), R_T, device=device)
    C = torch.zeros_like(R)
    return torch.stack([R, C])


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/others/receptor_binding/solve.py
"""Integrate binding with RK4: an association phase (ligand flowing) followed by a dissociation phase (buffer wash)."""

import torch

from problems._common.integrators import integrate

from .equation import rhs


def time_step(L: torch.Tensor, k_on: torch.Tensor, k_off: torch.Tensor) -> float:
    """Keep dt well below 1 / k_obs of the fastest (b, c) pair, so RK4 stays accurate."""
    k_fastest = (k_on.max() * L.max() + k_off.max()).item()
    return min(0.05, 0.5 / k_fastest)


def solve(
    y0: torch.Tensor,
    L: torch.Tensor,
    k_on: torch.Tensor,
    k_off: torch.Tensor,
    t_assoc: float,
    t_dissoc: float,
    depletion: bool,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times (T,), trajectory (T, 2, B, C)) with trajectory[:, 0] = R and trajectory[:, 1] = C.

    Every receptor variant b and every concentration c advances together in one (2, B, C) tensor.
    """
    dt = time_step(L, k_on, k_off)
    n_assoc = int(round(t_assoc / dt))
    n_dissoc = int(round(t_dissoc / dt))

    # 1) association: ligand at concentration L_c
    on = integrate(lambda y, t: rhs(y, L, k_on, k_off, depletion), y0, dt, n_assoc)
    # 2) dissociation: the buffer wash removes all free ligand, so complexes can only fall apart
    buffer = torch.zeros_like(L)
    off = integrate(lambda y, t: rhs(y, buffer, k_on, k_off, False), on[-1], dt, n_dissoc)

    traj = torch.cat([on, off[1:]])  # (T, 2, B, C)
    times = torch.arange(traj.shape[0], device=y0.device) * dt
    return times, traj


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/others/receptor_binding/plot.py
"""Animate binding: an SPR-style sensorgram, the binding isotherms θ(L), and receptors bound on a patch of cell."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

N_RECEPTORS = 16  # receptors drawn on each strip of membrane


def animate(
    times: torch.Tensor,
    theta: torch.Tensor,
    L: torch.Tensor,
    L_fine: torch.Tensor,
    theta_eq: torch.Tensor,
    names: list[str],
    t_assoc: float,
    path: Path,
) -> Path:
    stride = max(1, len(times) // 1500)  # RK4 takes small steps; ~1500 points are plenty for drawing
    t = times[::stride].cpu().numpy()
    th = theta[::stride].cpu().numpy()  # (T, B, C); the last column c = -1 is the sensorgram concentration
    conc = L.cpu().numpy()
    L_f, th_eq = L_fine.cpu().numpy(), theta_eq.cpu().numpy()
    B = th.shape[1]
    colors = [f"C{b}" for b in range(B)]

    fig = plt.figure(figsize=(8.2, 4.2))
    grid = fig.add_gridspec(2, 2, width_ratios=[1.25, 1], height_ratios=[1.35, 1])
    ax_s = fig.add_subplot(grid[:, 0])
    ax_i = fig.add_subplot(grid[0, 1])
    ax_r = fig.add_subplot(grid[1, 1])

    # --- sensorgram: θ_b(t) at one ligand concentration ---
    ax_s.set_xlim(0, t[-1])
    ax_s.set_ylim(0, 1.12)
    ax_s.set_xlabel("time (s)")
    ax_s.set_ylabel(r"fraction bound  $\theta = C / R_T$")
    ax_s.set_title(f"SPR sensorgram, L = {_conc_label(conc[-1])}")
    ax_s.axvline(t_assoc, color="#9ca3af", ls="--", lw=1)
    ax_s.text(t_assoc * 0.5, 1.1, "ligand on", ha="center", va="top", fontsize=8, color="#6b7280")
    ax_s.text(t_assoc + (t[-1] - t_assoc) * 0.5, 1.1, "buffer wash", ha="center", va="top", fontsize=8, color="#6b7280")
    traces = [ax_s.plot([], [], lw=1.8, color=colors[b], label=names[b])[0] for b in range(B)]
    ax_s.legend(loc="center left", bbox_to_anchor=(0.1, 0.66), fontsize=7.5)
    clock = ax_s.text(0.02, 0.06, "", transform=ax_s.transAxes, fontsize=9)

    # --- isotherms: equilibrium θ vs log L (dashed), current θ_bc(t) (dots), half point marked ---
    ax_i.set_xscale("log")
    ax_i.set_xlim(L_f[0], L_f[-1])
    ax_i.minorticks_off()  # fewer tick artists to redraw in every frame
    ax_i.set_xticks([1e-2, 1, 1e2, 1e4], ["0.01", "1", "100", "10⁴"])  # plain labels: no mathtext per frame
    ax_i.set_ylim(0, 1.05)
    ax_i.set_xlabel("ligand L (nM)", labelpad=1)
    ax_i.set_ylabel("θ")
    ax_i.set_title("binding isotherms", fontsize=10)
    for b in range(B):
        ax_i.plot(L_f, th_eq[b], "--", lw=1, color=colors[b])
        if th_eq[b, 0] <= 0.5 <= th_eq[b, -1]:  # where θ = ½: at L = K_D (at K_D + R_T/2 with depletion)
            ax_i.plot([np.interp(0.5, th_eq[b], L_f)], [0.5], "|", ms=9, mew=1.5, color=colors[b])
    ax_i.text(0.03, 0.9, "| : half bound, L = $K_D$", transform=ax_i.transAxes, fontsize=7.5)
    dots = [ax_i.plot([], [], "o", ms=3, color=colors[b])[0] for b in range(B)]

    # --- receptors on a patch of membrane (sensorgram concentration) ---
    ax_r.set_xlim(-0.8, N_RECEPTORS + 2.2)
    ax_r.set_ylim(-0.6, B - 0.1)
    ax_r.axis("off")
    ax_r.set_title(f"{N_RECEPTORS} receptors per cell patch", fontsize=10)
    x = np.arange(N_RECEPTORS)
    for b in range(B):
        ax_r.plot([-0.6, N_RECEPTORS - 0.4], [B - 1 - b - 0.25] * 2, color="#374151", lw=1.5)  # membrane
        ax_r.plot(*_receptor_shapes(x, B - 1 - b), color="#9ca3af", lw=1.5)
    ligands = [ax_r.plot([], [], "o", ms=4.5, color=colors[b])[0] for b in range(B)]
    counts = [ax_r.text(N_RECEPTORS, B - 1 - b, "", fontsize=8, va="center", color=colors[b]) for b in range(B)]

    def update(step: int):
        for b in range(B):
            traces[b].set_data(t[: step + 1], th[: step + 1, b, -1])
            dots[b].set_data(conc[:-1], th[step, b, :-1])
            n_bound = int(round(th[step, b, -1] * N_RECEPTORS))
            ligands[b].set_data(x[:n_bound], [B - 1 - b + 0.32] * n_bound)
            counts[b].set_text(f"{n_bound}/{N_RECEPTORS}")
        clock.set_text(f"t = {t[step]:.0f} s")

    fig.tight_layout(rect=(0, 0, 0.98, 1))  # a little room for the last tick label
    return save_gif(fig, update, frame_indices(len(t), 90), path)


def _receptor_shapes(x: np.ndarray, y: float) -> tuple[np.ndarray, np.ndarray]:
    """One polyline of Y-shaped receptors (stem + two arms) at heights around y, separated by NaN gaps."""
    xs, ys = [], []
    for xi in x:
        xs += [xi, xi, np.nan, xi - 0.3, xi, xi + 0.3, np.nan]
        ys += [y - 0.25, y, np.nan, y + 0.22, y, y + 0.22, np.nan]
    return np.array(xs), np.array(ys)


def _conc_label(nM: float) -> str:
    return f"{nM / 1e3:.3g} µM" if nM >= 1e3 else f"{nM:.3g} nM"


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'log10_L': 0.0,
    't_assoc': 120.0,
    't_dissoc': 240.0,
    'depletion': 'off',
    'R_T': 10.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.others.receptor_binding")]:
    del sys.modules[name]
main = importlib.import_module("problems.others.receptor_binding.main")
problem_dir = Path("problems/others/receptor_binding")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
